# Open-weight model test for the still workflow

Runs the **same inputs** (prompt, negative, cropped look-sheet refs) that the approved `nano-banana-pro` recipe used, on open models,
to compare two-character consistency and hands/feet. Fictional adult characters only.

Run: **Runtime > Run all**. A progress log and a small contact sheet are posted to a temporary notification topic so the agent can read the result.

In [ ]:
import subprocess, sys, traceback, urllib.request, os
TOPIC = "cmc-af291fc2ecbf"
def notify(msg, title="colab", file=None):
    try:
        hdr = {"Title": title}
        data = msg.encode()[:3800]
        if file:
            hdr["Filename"] = os.path.basename(file); data = open(file, "rb").read()
            hdr["Message"] = msg[:200]
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{TOPIC}", data=data, headers=hdr, method="PUT" if file else "POST"), timeout=60)
    except Exception as e:
        print("notify failed:", e)
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
print(gpu); notify("start: " + gpu)

In [ ]:
%%bash
set -e
cd /content
[ -d new ] || git clone --depth 1 --branch cursor/ai-realistic-comic-e63b https://github.com/hailinsu-create/new.git
pip -q install pydantic-settings python-dotenv typer fal-client pyyaml 2>&1 | tail -2
pip -q install -U "git+https://github.com/huggingface/diffusers" transformers accelerate bitsandbytes sentencepiece torchao huggingface_hub 2>&1 | tail -2

In [ ]:
import json, sys
os.chdir("/content/new/ai-realistic-comic")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
from pathlib import Path
try:
    from open_model_job import build_job
    STILLS = ["hades-persephone-throne", "baisuzhen-xuxian-coil"]
    JOBS = {s: build_job(s, 0.55, Path(f"/content/work/{s}")) for s in STILLS}
    notify("jobs built: " + ", ".join(JOBS))
except Exception:
    notify("JOB BUILD FAILED\n" + traceback.format_exc()[-3000:]); raise

In [ ]:
import torch, gc
from PIL import Image
OUT = Path("/content/out"); OUT.mkdir(exist_ok=True)
W, H = 960, 1280
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM {vram:.0f} GB")

def free():
    gc.collect(); torch.cuda.empty_cache()

def run_job(pipe, name, job, seeds=(1, 2), **kw):
    imgs = [Image.open(p).convert("RGB") for p in job["refs"]]
    for seed in seeds:
        g = torch.Generator("cuda").manual_seed(seed)
        r = pipe(image=imgs, prompt=job["prompt"], negative_prompt=job["negative"], height=H, width=W, generator=g, **kw)
        path = OUT / f"{job['still_id']}__{name}__s{seed}.png"
        r.images[0].save(path); print("saved", path); notify(f"saved {path.name}")

## Model 1: Qwen-Image-Edit-2511 (Apache 2.0, needs A100-class GPU)

In [ ]:
if vram < 40:
    notify(f"Qwen-Image-Edit-2511 skipped: {vram:.0f} GB VRAM")
    print("skipped")
else:
    try:
        from diffusers import QwenImageEditPlusPipeline
        kw = dict(torch_dtype=torch.bfloat16)
        if vram < 70:
            from diffusers.quantizers import PipelineQuantizationConfig
            kw["quantization_config"] = PipelineQuantizationConfig(
                quant_backend="bitsandbytes_4bit",
                quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16},
                components_to_quantize=["transformer", "text_encoder"])
        pipe = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2511", **kw)
        pipe.enable_model_cpu_offload()
        for j in JOBS.values():
            run_job(pipe, "qwen2511", j, true_cfg_scale=4.0, guidance_scale=1.0, num_inference_steps=40)
        del pipe; free()
    except Exception:
        notify("QWEN FAILED\n" + traceback.format_exc()[-3000:]); traceback.print_exc()

## Model 2: FLUX.2 klein 4B (Apache 2.0)

In [ ]:
try:
    from diffusers import Flux2KleinPipeline
    pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    notify("flux2 klein loaded")
    for j in JOBS.values():
        imgs = [Image.open(p).convert("RGB") for p in j["refs"]]
        for seed in (1, 2):
            g = torch.Generator("cuda").manual_seed(seed)
            im = pipe(image=imgs, prompt=j["prompt"], height=H, width=W, num_inference_steps=4, guidance_scale=1.0, generator=g).images[0]
            p = OUT / f"{j['still_id']}__flux2klein4b__s{seed}.png"; im.save(p); print("saved", p); notify(f"saved {p.name}")
    del pipe; free()
except Exception:
    v = subprocess.run("pip list 2>/dev/null | grep -i -E '^(diffusers|transformers|torch|torchvision|huggingface-hub|peft|accelerate|bitsandbytes|numpy|torchao) '", shell=True, capture_output=True, text=True).stdout
    notify("FLUX FAILED\n" + traceback.format_exc()[-2500:] + "\n" + v); traceback.print_exc()

## Contact sheet: approved nano-banana-pro vs open models

In [ ]:
import glob
for s, j in JOBS.items():
    files = [j["approved_image"]] + sorted(glob.glob(str(OUT / f"{s}__*.png")))
    tiles = [Image.open(f).convert("RGB") for f in files]
    th = 640
    tiles = [t.resize((int(t.width * th / t.height), th)) for t in tiles]
    sheet = Image.new("RGB", (sum(t.width for t in tiles), th), "white")
    x = 0
    for t in tiles: sheet.paste(t, (x, 0)); x += t.width
    path = f"/content/sheet_{s}.jpg"; sheet.save(path, quality=85)
    display(sheet)
    notify(f"{s}: first tile = APPROVED nano-banana-pro, then " + ", ".join(Path(f).stem.split("__", 1)[-1] for f in files[1:]), file=path)
notify("DONE")